# 6.1 模型版本管理与注册中心

产业级AI部署中，模型版本管理是基础设施而非可选项。一个7B模型从训练到部署涉及数十个版本（不同量化、不同格式、不同硬件适配），没有系统化的版本管理将导致混乱。本章建立模型版本管理的完整方法论。

## 为什么需要模型版本管理？

| 没有版本管理 | 有版本管理 |
|------------|----------|
| "这个模型是哪个版本的？" → 无人知晓 | 每个模型有唯一版本号和元数据 |
| 线上出问题 → 不知道回滚到哪个版本 | 一键回滚到上一个稳定版本 |
| 新模型上线 → 不确定是否比旧的好 | A/B测试+灰度发布，数据驱动决策 |
| 多平台部署 → 版本不一致 | 统一注册中心，各平台拉取指定版本 |
| 合规审计 → 无法追溯模型变更 | 完整的变更历史和审批记录 |

### 模型版本的复杂性

一个模型在产业级部署中会产生大量衍生版本：

```
Qwen2.5-7B-Instruct (基座模型 v1.0)
├── Q4_K_M.gguf          (llama.cpp CPU版)
├── Q5_K_M.gguf          (llama.cpp CPU版, 高精度)
├── onnx_int8/            (ONNX INT8, 通用中转)
├── qnn_int4.bin          (高通NPU版)
├── coreml_fp16.mlpackage (Apple ANE版)
├── openvino_int8.xml     (Intel NPU版)
├── lora_task_v2.safetensors (LoRA适配器 v2)
└── lora_task_v3.safetensors (LoRA适配器 v3)
```

每个衍生版本都需要独立管理、测试和部署。

## 6.1.1 模型版本命名规范

### 语义化版本号

```
模型版本: {model_name}-{major}.{minor}.{patch}-{quant}-{format}-{target}

示例:
  qwen2.5-7b-1.0.0-fp32-pytorch-origin
  qwen2.5-7b-1.0.0-q4km-gguf-cpu
  qwen2.5-7b-1.0.0-int8-onnx-qnn
  qwen2.5-7b-1.0.0-fp16-coreml-ane
  qwen2.5-7b-1.1.0-q4km-gguf-cpu    (minor: 功能更新)
  qwen2.5-7b-1.0.1-q4km-gguf-cpu    (patch: bug修复)
```

### 版本号规则

| 字段 | 含义 | 变更条件 | 示例 |
|------|------|---------|------|
| **major** | 主版本 | 模型架构变更/基座模型升级 | 1→2 |
| **minor** | 次版本 | 微调更新/量化方案优化 | 0→1 |
| **patch** | 补丁 | bug修复/编译选项调整 | 0→1 |
| **quant** | 量化 | 量化配置 | q4km/int8/fp16 |
| **format** | 格式 | 模型格式 | gguf/onnx/coreml |
| **target** | 目标 | 目标硬件 | cpu/qnn/ane |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from datetime import datetime

@dataclass
class ModelVersion:
    model_name: str
    major: int
    minor: int
    patch: int
    quant: str
    format: str
    target: str
    base_model_id: Optional[str] = None
    created_at: str = ''
    size_mb: float = 0
    checksum: str = ''
    metrics: Dict = field(default_factory=dict)
    status: str = 'draft'

    @property
    def version_str(self):
        return f"{self.model_name}-{self.major}.{self.minor}.{self.patch}-{self.quant}-{self.format}-{self.target}"

    @property
    def semver(self):
        return f"{self.major}.{self.minor}.{self.patch}"

class ModelRegistry:
    """模型注册中心"""
    def __init__(self):
        self.versions: Dict[str, ModelVersion] = {}
        self.lineage: Dict[str, List[str]] = {}

    def register(self, version: ModelVersion):
        vid = version.version_str
        version.created_at = datetime.now().isoformat()
        self.versions[vid] = version
        if version.base_model_id:
            if version.base_model_id not in self.lineage:
                self.lineage[version.base_model_id] = []
            self.lineage[version.base_model_id].append(vid)

    def promote(self, version_id, stage):
        valid_stages = ['draft', 'staging', 'canary', 'production']
        if version_id in self.versions and stage in valid_stages:
            self.versions[version_id].status = stage

    def get_production_version(self, model_name, quant, format, target):
        for vid, v in self.versions.items():
            if (v.model_name == model_name and v.quant == quant and
                v.format == format and v.target == target and v.status == 'production'):
                return v
        return None

    def rollback(self, model_name, quant, format, target):
        current = self.get_production_version(model_name, quant, format, target)
        if current is None:
            return None
        candidates = [v for v in self.versions.values()
                      if (v.model_name == model_name and v.quant == quant and
                          v.format == format and v.target == target and
                          v.status in ['staging', 'canary'])]
        if not candidates:
            older = [v for v in self.versions.values()
                     if (v.model_name == model_name and v.quant == quant and
                         v.format == format and v.target == target and
                         (v.major, v.minor, v.patch) < (current.major, current.minor, current.patch))]
            candidates = sorted(older, key=lambda v: (v.major, v.minor, v.patch), reverse=True)
        if candidates:
            current.status = 'rolled_back'
            candidates[0].status = 'production'
            return candidates[0]
        return None

registry = ModelRegistry()
base = ModelVersion('qwen2.5-7b', 1, 0, 0, 'fp32', 'pytorch', 'origin',
                     size_mb=28000, metrics={'ppl': 5.12})
registry.register(base)

derivatives = [
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'q4km', 'gguf', 'cpu',
                 base_model_id=base.version_str, size_mb=4200, metrics={'ppl': 5.45, 'tps': 18}),
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'int8', 'onnx', 'qnn',
                 base_model_id=base.version_str, size_mb=7800, metrics={'ppl': 5.38, 'tps': 22}),
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'fp16', 'coreml', 'ane',
                 base_model_id=base.version_str, size_mb=14000, metrics={'ppl': 5.15, 'tps': 15}),
    ModelVersion('qwen2.5-7b', 1, 1, 0, 'q4km', 'gguf', 'cpu',
                 base_model_id=base.version_str, size_mb=4200, metrics={'ppl': 5.30, 'tps': 18}),
]
for d in derivatives:
    registry.register(d)

registry.promote(derivatives[0].version_str, 'production')
registry.promote(derivatives[3].version_str, 'staging')

print("=== 模型注册中心 ===")
print(f"\n{'版本ID':<55} {'状态':<12} {'大小(MB)':<10} {'PPL':<8} {'TPS'}")
print("-" * 95)
for vid, v in registry.versions.items():
    ppl = v.metrics.get('ppl', '-')
    tps = v.metrics.get('tps', '-')
    print(f"{vid:<55} {v.status:<12} {v.size_mb:<10.0f} {ppl:<8} {tps}")

print(f"\n=== 血缘关系 ===")
for parent, children in registry.lineage.items():
    print(f"{parent}")
    for child in children:
        print(f"  └── {child}")

## 6.1.2 模型部署生命周期

### 四阶段发布流程

```
Draft → Staging → Canary → Production
  │        │         │         │
  │        │         │         └── 100%流量
  │        │         └── 1-5%流量验证
  │        └── 完整测试通过
  └── 初始版本，未测试

回滚路径: Production → Canary → Staging (任一阶段发现问题)
```

### 各阶段的验证要求

| 阶段 | 验证内容 | 通过标准 | 停留时间 |
|------|---------|---------|----------|
| **Draft→Staging** | 精度验证+性能基准 | PPL增加<0.5, 吞吐>基线90% | 自动 |
| **Staging→Canary** | 集成测试+兼容性 | 所有测试用例通过 | 1-2天 |
| **Canary→Production** | 线上小流量验证 | 错误率<0.1%, 延迟P99<SLA | 3-7天 |
| **回滚** | 发现问题 | 任何指标超阈值 | 立即 |

In [ ]:
class DeploymentPipeline:
    """模型部署流水线"""
    STAGES = ['draft', 'staging', 'canary', 'production']

    def __init__(self, registry: ModelRegistry):
        self.registry = registry
        self.validation_results = {}

    def validate_precision(self, version_id, baseline_ppl, threshold=0.5):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'pass': False, 'reason': 'version not found'}
        model_ppl = v.metrics.get('ppl', float('inf'))
        ppl_increase = model_ppl - baseline_ppl
        return {'pass': ppl_increase < threshold, 'ppl_increase': ppl_increase, 'threshold': threshold}

    def validate_performance(self, version_id, baseline_tps, min_ratio=0.9):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'pass': False, 'reason': 'version not found'}
        model_tps = v.metrics.get('tps', 0)
        ratio = model_tps / baseline_tps if baseline_tps > 0 else 0
        return {'pass': ratio >= min_ratio, 'tps_ratio': ratio, 'threshold': min_ratio}

    def promote_with_validation(self, version_id, baseline_ppl, baseline_tps):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'success': False, 'reason': 'version not found'}
        current_stage = v.status
        current_idx = self.STAGES.index(current_stage) if current_stage in self.STAGES else -1
        if current_idx >= len(self.STAGES) - 1:
            return {'success': False, 'reason': 'already in production'}
        precision = self.validate_precision(version_id, baseline_ppl)
        performance = self.validate_performance(version_id, baseline_tps)
        if not precision['pass'] or not performance['pass']:
            return {'success': False, 'precision': precision, 'performance': performance}
        next_stage = self.STAGES[current_idx + 1]
        self.registry.promote(version_id, next_stage)
        return {'success': True, 'from': current_stage, 'to': next_stage,
                'precision': precision, 'performance': performance}

pipeline = DeploymentPipeline(registry)

print("=== 模型部署流水线模拟 ===")
staging_id = [v.version_str for v in registry.versions.values() if v.status == 'staging'][0]
print(f"\n尝试将 {staging_id} 从 staging 提升到 canary...")
result = pipeline.promote_with_validation(staging_id, baseline_ppl=5.45, baseline_tps=18)
if result['success']:
    print(f"  ✓ 提升成功: {result['from']} → {result['to']}")
    print(f"    精度验证: PPL增加={result['precision']['ppl_increase']:.2f} (阈值<{result['precision']['threshold']})")
    print(f"    性能验证: TPS比例={result['performance']['tps_ratio']:.2f} (阈值>{result['performance']['threshold']})")
else:
    print(f"  ✗ 提升失败")
    if 'precision' in result:
        print(f"    精度: PPL增加={result['precision']['ppl_increase']:.2f}")
    if 'performance' in result:
        print(f"    性能: TPS比例={result['performance']['tps_ratio']:.2f}")

print(f"\n=== 模型版本管理总结 ===")
print(f"1. 语义化版本号: model-major.minor.patch-quant-format-target")
print(f"2. 四阶段发布: Draft → Staging → Canary → Production")
print(f"3. 每阶段必须通过精度+性能验证才能提升")
print(f"4. 血缘追踪: 每个衍生版本记录其基座模型")
print(f"5. 一键回滚: 发现问题立即回退到上一稳定版本")
print(f"6. 自动化CI/CD: 模型更新自动触发验证+部署流水线")